In [0]:
# =============================================
# INSIDER THREAT DETECTION - ML-förberedelser
# =============================================

print("🔧 Steg 1: Installerar bibliotek (om det behövs)...")
%pip install snowflake-connector-python pandas scikit-learn -q

print("📚 Steg 2: Importerar bibliotek...")
import snowflake.connector as sf
import pandas as pd
from getpass import getpass

# =============================================
# Steg 3: Anslut till Snowflake (lösenord skrivs i terminalen)
# =============================================
print("\n🔐 Steg 3: Ansluter till Snowflake...")
print("Ange ditt Snowflake-lösenord (syns inte på skärmen):")

conn = sf.connect(
    account='jnqaoob-to29395',
    user='ARSENIY',
    password=getpass(),  # ← Frågar i terminalen, syns inte!
    warehouse='COMPUTE_WH',
    database='INSIDER_THREAT_DB',
    schema='DBT_AB'
)
print("✅ Anslutning lyckades!")

# =============================================
# Steg 4: Hämta data
# =============================================
print("\n📊 Steg 4: Hämtar data från Snowflake...")

# Kontrollera antal rader först
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM FCT_FEATURES")
row_count = cursor.fetchone()[0]
print(f"   📍 Hittade {row_count:,} rader i FCT_FEATURES")

# Hämta all data
query = "SELECT * FROM FCT_FEATURES"
df = pd.read_sql(query, conn)
print(f"✅ Hämtade {len(df):,} rader och {len(df.columns)} kolumner")

# Stäng anslutningen (viktigt för att undvika timeout)
conn.close()
print("🔌 Anslutning stängd.")

# =============================================
# Steg 5: Förbered data för ML
# =============================================
print("\n🤖 Steg 5: Förbereder data för ML...")

# Välj features (X) och target (y)
y = df['ATTACK_TYPE']  # Målvariabel

# Välj bara numeriska kolumner för ML
numerical_cols = df.select_dtypes(include=['number']).columns
X = df[numerical_cols]

print(f"   📊 Features: {len(X.columns)} numeriska kolumner")
print(f"   🎯 Target: {y.nunique()} unika attacktyper")

# Visa de första raderna (valfritt)
print("\n📋 Första 5 raderna av datan:")
print(df.head())

print("\n✅ Allt är klart! Fortsätt till ML-modellen.")

🔧 Steg 1: Installerar bibliotek (om det behövs)...
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
📚 Steg 2: Importerar bibliotek...

🔐 Steg 3: Ansluter till Snowflake...
Ange ditt Snowflake-lösenord (syns inte på skärmen):


 [REDACTED]

✅ Anslutning lyckades!

📊 Steg 4: Hämtar data från Snowflake...
   📍 Hittade 4,000 rader i FCT_FEATURES


/home/spark-570f300d-4dc2-4876-b18d-67/.ipykernel/3037/command-8161561072563701-1834520868:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


✅ Hämtade 4,000 rader och 18 kolumner
🔌 Anslutning stängd.

🤖 Steg 5: Förbereder data för ML...
   📊 Features: 14 numeriska kolumner
   🎯 Target: 8 unika attacktyper

📋 Första 5 raderna av datan:
        USER_SESSION_ID          ATTACK_TYPE  ...  RISK_CATEGORY  FLAG_RISK_SCORE
0   EHR_patient_records  unauthorized_access  ...         normal                0
1  PACS_patient_records   credential_sharing  ...      high_risk                2
2       HL7_API_billing  unauthorized_access  ...      high_risk                2
3        PACS_radiology               normal  ...      high_risk                2
4    PACS_prescriptions    data_modification  ...      high_risk                2

[5 rows x 18 columns]

✅ Allt är klart! Fortsätt till ML-modellen.


In [0]:
# =============================================
# STEG 9: IMPORTERA MLflow
# =============================================

print("📦 Steg 9: Importerar MLflow...")

# Importera MLflow-bibliotek
import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn
from getpass import getpass

print(f"✅ MLflow version: {mlflow.__version__}")

# =============================================
# Fråga efter e-postadress (för experiment-namn)
# =============================================
print("\n📧 Ange din e-postadress (samma som i Databricks):")
print("   (Den syns inte på skärmen – skriv och tryck Enter)")

user_email = getpass()

# Skapa experiment-namn baserat på din e-post
experiment_name = f"/Users/{user_email}/insider_threat_detection"

print(f"\n🔍 Experiment-namn: {experiment_name}")

# =============================================
# Skapa eller hämta experimentet
# =============================================
try:
    mlflow.create_experiment(experiment_name)
    print(f"✅ Nytt experiment skapat!")
except:
    # Experimentet finns redan – det är OK!
    print(f"✅ Experimentet finns redan!")

# Sätt experimentet som aktivt
mlflow.set_experiment(experiment_name)
print("✅ Experimentet är nu aktivt!")

# =============================================
# Visa var du hittar dina experiment
# =============================================
print("\n📊 MLflow är redo! Fortsätt till Steg 10 för att bygga modellen.")
print("\n💡 Tips: Du hittar dina experiment i Databricks via:")
print("   → Menyn till vänster → 'Experiment' → 'insider_threat_detection'")

📦 Steg 9: Importerar MLflow...
✅ MLflow version: 3.8.1

📧 Ange din e-postadress (samma som i Databricks):
   (Den syns inte på skärmen – skriv och tryck Enter)


 [REDACTED]


🔍 Experiment-namn: /Users/[REDACTED]/insider_threat_detection
✅ Nytt experiment skapat!
✅ Experimentet är nu aktivt!

📊 MLflow är redo! Fortsätt till Steg 10 för att bygga modellen.

💡 Tips: Du hittar dina experiment i Databricks via:
   → Menyn till vänster → 'Experiment' → 'insider_threat_detection'


In [0]:
print(f"X har {X.shape[0]} rader och {X.shape[1]} kolumner")
print(f"y har {len(y)} rader")
print(f"Attacktyper: {y.unique()}")

X har 4000 rader och 14 kolumner
y har 4000 rader
Attacktyper: ['unauthorized_access' 'credential_sharing' 'normal' 'data_modification'
 'audit_log_tampering' 'policy_violation' 'privilege_misuse'
 'insider_data_leakage']
